# Preprocessing Notebook

- mood별 MIDI 파일을 읽어 학습용 piano-roll tensor로 변환한다.
- 4/4 박자 MIDI만 사용하고, 2마디 단위 32-step chunk로 나눈다.
- 결과는 `midivae/data/{mood}` 아래 `.pt` 파일로 저장한다.


## 0. 설치와 라이브러리 준비
MIDI 파싱과 tensor 저장에 필요한 패키지를 설치하고 라이브러리를 불러온다.


In [ ]:
!pip -q install pretty_midi

import os

import numpy as np
import pretty_midi
import torch


## 1. Google Drive 마운트와 원본 압축 해제
Drive에 있는 categorized MIDI 압축 파일을 Colab 로컬 작업 공간에 풀어 전처리 속도를 높인다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/ML/midivae/categorized_midi.zip"
UNZIP_DIR = "/content/categorized_midi"

# Drive의 원본 ZIP을 Colab 로컬 디스크에 한 번만 압축 해제한다.
if not os.path.exists(UNZIP_DIR):
    print(f"📦 압축 해제 시작: {ZIP_PATH} -> {UNZIP_DIR}")
    !unzip -q "{ZIP_PATH}" -d "{UNZIP_DIR}"
    print("✅ 압축 해제 완료!")
else:
    print("✅ 이미 압축이 해제되어 있습니다.")


## 2. 전처리 경로와 기준 설정
원본 MIDI 경로, 저장 경로, mood 목록, 최소 chunk 수와 mood별 최대 처리 개수를 지정한다.


In [ ]:
RAW_DATA_ROOT = "/content/categorized_midi/categorized_midi"
SAVE_DATA_ROOT = "/content/drive/MyDrive/ML/midivae/data"

MOODS = [
    'angry', 'exciting', 'fear', 'funny', 'happy',
    'lazy', 'magnificent', 'quiet', 'romantic', 'sad', 'warm'
]

# 학습 시퀀스 길이가 8이므로 next-step 예측에는 최소 9개 chunk가 필요하다.
MIN_CHUNKS_REQUIRED = 9

# mood별로 저장할 최대 전처리 파일 수
MAX_FILES_PER_MOOD = 1000


## 3. MIDI를 학습용 tensor로 변환
각 MIDI를 4채널 piano-roll로 바꾸고 32-step chunk 묶음으로 저장한다.


In [ ]:
def process_midi_dataset():
    total_success = 0
    total_skipped = 0

    print(f"🚀 전처리 시작: {RAW_DATA_ROOT} -> {SAVE_DATA_ROOT}")
    print(f"🎯 각 감정별 목표 최대 처리 개수: {MAX_FILES_PER_MOOD}개\n")

    for mood in MOODS:
        raw_mood_dir = os.path.join(RAW_DATA_ROOT, mood)
        save_mood_dir = os.path.join(SAVE_DATA_ROOT, mood)
        os.makedirs(save_mood_dir, exist_ok=True)

        if not os.path.exists(raw_mood_dir):
            print(f"⚠️ {mood} 폴더를 찾을 수 없어 건너뜁니다.")
            continue

        midi_files = [f for f in os.listdir(raw_mood_dir) if f.lower().endswith(('.mid', '.midi'))]
        print(f"🎧 [{mood.upper()}] 폴더 처리 중... (탐색 대상 파일 {len(midi_files)}개)")

        success_count = 0
        skipped_count = 0

        for filename in midi_files:
            if success_count >= MAX_FILES_PER_MOOD:
                print(f"   ✅ 목표치 {MAX_FILES_PER_MOOD}개 달성! 다음 감정으로 넘어갑니다.")
                break

            filepath = os.path.join(raw_mood_dir, filename)
            name_no_ext = os.path.splitext(filename)[0]

            try:
                pm = pretty_midi.PrettyMIDI(filepath)

                # 4/4 박자 MIDI만 학습 데이터로 사용한다.
                is_four_four = (
                    pm.time_signature_changes
                    and pm.time_signature_changes[0].numerator == 4
                    and pm.time_signature_changes[0].denominator == 4
                )
                if not is_four_four:
                    skipped_count += 1
                    continue

                bpm = pm.estimate_tempo()
                if bpm <= 0:
                    skipped_count += 1
                    continue

                # 16 position/bar 기준으로 2마디를 32 step chunk로 만든다.
                step_duration = (60.0 / bpm) / 4.0
                total_steps = int(pm.get_end_time() / step_duration)
                chunk_steps = 32

                if total_steps < chunk_steps:
                    skipped_count += 1
                    continue

                piano_roll = np.zeros((4, 88, total_steps), dtype=np.uint8)
                for inst in pm.instruments:
                    # 0: melody, 1: harmony, 2: bass, 3: drum
                    channel = 3 if inst.is_drum else (
                        2 if 32 <= inst.program <= 39 else (
                            1 if (40 <= inst.program <= 55) or (88 <= inst.program <= 95) else 0
                        )
                    )

                    for note in inst.notes:
                        if 21 <= note.pitch <= 108:
                            start = int(note.start / step_duration)
                            end = max(start + 1, int(note.end / step_duration))
                            end = min(end, total_steps)
                            if start < total_steps:
                                piano_roll[channel, note.pitch - 21, start:end] = 1

                chunks = []
                for start in range(0, total_steps - chunk_steps + 1, chunk_steps):
                    end = start + chunk_steps
                    chunks.append(piano_roll[:, :, start:end])

                if len(chunks) >= MIN_CHUNKS_REQUIRED:
                    save_path = os.path.join(save_mood_dir, f"{name_no_ext}.pt")
                    torch.save({"chunks": torch.tensor(np.array(chunks), dtype=torch.uint8)}, save_path)
                    success_count += 1
                else:
                    skipped_count += 1

            except Exception:
                skipped_count += 1

        print(f"   -> 완료: 저장 {success_count}곡 | 스킵 {skipped_count}곡")
        total_success += success_count
        total_skipped += skipped_count

    print("============================================================")
    print(f"🎉 전체 전처리 완료! 총 {total_success}곡이 준비되었습니다.")
    print("============================================================")


process_midi_dataset()
